# Практика · Постановка детекції

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі дані зошит генерує формулами, ваги не завантажуються.
> Досить `torch`, `torchvision`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає один детектор. Заміряно: **близько 70 секунд** на чотирьох
> ядрах без відеокарти. Найдовше йде саме навчання — 35-40 секунд.

Що зробимо:

1. напишемо **свій IoU** і звіримо з `torchvision.ops.box_iou`;
2. покажемо три різні помилки з однаковим IoU;
3. порахуємо, на скільки пікселів можна зсунути рамку при різних порогах;
4. згенеруємо сцени 64×64 й побачимо дисбаланс «предмет проти фону» в числах;
5. навчимо детектор і подивимось, скільки рамок він видає на 247 справжніх;
6. напишемо **свій NMS** і звіримо з `ops.nms` на тих самих входах;
7. напишемо **свій AP** і звіримо з ручним підрахунком на крихітному прикладі;
8. дістанемо `mAP@0.5` і `mAP@0.5:0.95` і побачимо різницю.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torchvision.ops import box_iou, nms, box_convert
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)
torch.manual_seed(0)

print("torch      ", torch.__version__)
print("numpy      ", np.__version__)
print("потоків    ", torch.get_num_threads())

## 1 · IoU власноруч

IoU — це площа перетину двох рамок, поділена на площу їхнього обʼєднання. Рамку
скрізь записуємо у форматі `xyxy`: лівий-верхній кут і правий-нижній.

Перетин двох прямокутників — теж прямокутник. Його ліва межа — **правіша** з двох
лівих меж, права — **лівіша** з двох правих. Якщо після цього ширина вийшла
відʼємною, рамки просто не перетинаються, і площа дорівнює нулю.

In [ ]:
def iou_by_hand(box_a, box_b):
    """IoU двох рамок у форматі xyxy. Той самий розрахунок, що в бібліотеці."""
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])

    # відʼємна сторона означає «не перетинаються», тому зрізаємо її нулем
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    # спільну частину при додаванні площ порахували двічі — віднімаємо один раз
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0, intersection, union


first = [10.0, 10.0, 30.0, 30.0]
second = [20.0, 20.0, 40.0, 40.0]
value, intersection, union = iou_by_hand(first, second)

print("перетин    ", intersection)
print("обʼєднання ", union)
print("IoU руками  %.6f" % value)

Тепер те саме бібліотекою. `box_iou` приймає два тензори рамок і повертає матрицю
«кожна з кожною», тому для двох рамок беремо єдину клітинку `[0, 0]`.

In [ ]:
library_value = box_iou(torch.tensor([first]), torch.tensor([second]))[0, 0].item()
print("IoU бібліотекою %.6f" % library_value)

assert abs(value - library_value) < 1e-6, "наш IoU розійшовся з box_iou!"
print("✅ збігається до шостого знака")

## 2 · Що IoU не розрізняє

Візьмімо квадрат 40×40 і три різні помилки, кожна з яких дає з ним IoU рівно 0.5:

- рамка **менша**: та сама середина, але сторона 40·√0.5 ≈ 28.3;
- рамка **зсунута**: та сама сторона 40, але зміщена на 40·(1−0.5)/(1+0.5) ≈ 13.3;
- рамка **навколо повернутого предмета**: сторона 40/√0.5 ≈ 56.6.

Формули беруться з рівняння «перетин ділити на обʼєднання дорівнює 0.5», розвʼязаного
відносно сторони або зсуву. Перевіримо їх обчисленням.

In [ ]:
target_iou = 0.5
truth = [0.0, 0.0, 40.0, 40.0]

smaller_side = 40 * np.sqrt(target_iou)
offset = 40 * (1 - target_iou) / (1 + target_iou)
bigger_side = 40 / np.sqrt(target_iou)

variants = {
    "менша":     [(40 - smaller_side) / 2, (40 - smaller_side) / 2,
                  (40 + smaller_side) / 2, (40 + smaller_side) / 2],
    "зсунута":   [offset, 0.0, 40 + offset, 40.0],
    "повернута": [(40 - bigger_side) / 2, (40 - bigger_side) / 2,
                  (40 + bigger_side) / 2, (40 + bigger_side) / 2],
}

for name, box in variants.items():
    got, _, _ = iou_by_hand(truth, box)
    side = box[2] - box[0]
    print("%-10s сторона %5.1f  зсув %5.1f  IoU %.4f" %
          (name, side, box[0] - truth[0], got))
    assert abs(got - target_iou) < 1e-9, "варіант «%s» дав не 0.5" % name

print("✅ три різні помилки — одне й те саме число")

## 3 · На скільки можна промахнутись

Для двох рамок однакового розміру, зсунутих уздовж однієї осі на `s`, IoU дорівнює
`(a − s) / (a + s)`, де `a` — сторона. Розвʼязавши це відносно `s`, дістаємо
допустимий зсув для заданого порогу. Саме звідси беруться числа розділу 08 лекції.

In [ ]:
side_length = 40.0
print("рамка зі стороною %.0f px" % side_length)
print()
for wanted in (0.50, 0.75, 0.90):
    shift = side_length * (1 - wanted) / (1 + wanted)
    moved = [shift, 0.0, side_length + shift, side_length]
    check, _, _ = iou_by_hand([0.0, 0.0, side_length, side_length], moved)
    print("IoU %.2f → зсув %5.1f px (%.0f %% сторони), перевірка IoU = %.4f"
          % (wanted, shift, 100 * shift / side_length, check))

## 4 · Формати рамки

Одна й та сама рамка записується трьома способами, і всі три — це просто чотири
числа. Помилка при переплутуванні **тиха**: винятку не буде, буде поганий
результат. `box_convert` перекладає між форматами.

In [ ]:
square = torch.tensor([[31.0, 40.0, 50.0, 59.0]])          # xyxy
as_xywh = box_convert(square, in_fmt="xyxy", out_fmt="xywh")
as_cxcywh = box_convert(square, in_fmt="xyxy", out_fmt="cxcywh")

print("xyxy   ", square.tolist()[0])
print("xywh   ", as_xywh.tolist()[0])
print("cxcywh ", as_cxcywh.tolist()[0])
print("нормовані (поділені на 64):",
      [round(v / 64, 3) for v in as_cxcywh.tolist()[0]])

# що буде, якщо xywh віддати функції, яка чекає xyxy
wrong = torch.tensor(as_xywh.tolist())
print()
print("IoU квадрата із самим собою, прочитаним як xyxy: %.4f"
      % box_iou(square, wrong)[0, 0].item())
print("Винятку немає — просто нуль. Саме так ця помилка й ховається.")

## 5 · Датасет: сцени 64×64

Кожна сцена — полотно 64 на 64 пікселі, від одного до трьох предметів трьох класів
(коло, квадрат, трикутник) радіусом 6-10 пікселів, шум зі стандартним відхиленням
0.12. Рамка кожного предмета рахується **з його маски**, тобто вона істинна за
побудовою: людина її не ставила й помилитись не могла.

In [ ]:
SIZE, GRID, CELL = 64, 8, 8
CLASS_NAMES = ["коло", "квадрат", "трикутник"]


def shape_mask(kind, center_x, center_y, radius):
    """Маска однієї фігури на полотні 64×64."""
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                   # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                   # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def make_scene(rng):
    """Одна сцена: картинка, рамки з масок, мітки класів."""
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels = [], []
    for _ in range(int(rng.integers(1, 4))):
        for _attempt in range(40):
            radius = int(rng.integers(6, 11))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            mask = shape_mask(kind, center_x, center_y, radius)
            ys, xs = np.nonzero(mask)
            # рамка беруться з маски: край + 1, як в угоді COCO
            box = [float(xs.min()), float(ys.min()),
                   float(xs.max() + 1), float(ys.max() + 1)]

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in boxes:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, np.array(boxes, np.float32), np.array(labels, np.int64)


def make_dataset(seed, count):
    rng = np.random.default_rng(seed)
    return [make_scene(rng) for _ in range(count)]


started = time.time()
train_set = make_dataset(42, 400)
test_set = make_dataset(7, 120)
true_box_count = sum(len(scene[1]) for scene in test_set)

print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, перевірних %d" % (len(train_set), len(test_set)))
print("істинних рамок у перевірному наборі: %d" % true_box_count)

Подивимось на перші три перевірні сцени разом з їхніми істинними рамками.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for axis, (image, boxes, labels) in zip(axes, test_set[:3]):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    for box, label in zip(boxes, labels):
        axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                     fill=False, edgecolor="#0f766e", linewidth=1.6))
        axis.text(box[0], box[1] - 1, CLASS_NAMES[label], color="#0f766e", fontsize=7)
    axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout()
plt.show()
print("бірюзові прямокутники — еталонна розмітка, порахована з масок")

## 6 · Дисбаланс у числах

Розібʼємо кожну сцену сіткою 8 на 8 і позначимо позитивною кожну клітинку, чий
**центр** потрапив усередину якоїсь рамки. Ця сама розмітка стане ціллю для
навчання. Заразом порахуємо, скільки клітинок припадає на фон.

In [ ]:
def encode_targets(boxes, labels):
    """Ціль детектора: обʼєктність, чотири числа рамки й клас на кожну клітинку."""
    objectness = np.zeros((GRID, GRID), np.float32)
    box_target = np.zeros((4, GRID, GRID), np.float32)
    class_target = np.zeros((GRID, GRID), np.int64)
    # якщо клітинка потрапила всередину двох рамок, вона відповідає за меншу
    smallest_area = np.full((GRID, GRID), 1e9, np.float32)

    for (x1, y1, x2, y2), label in zip(boxes, labels):
        box_area = (x2 - x1) * (y2 - y1)
        for row in range(GRID):
            for col in range(GRID):
                point_x = col * CELL + CELL / 2
                point_y = row * CELL + CELL / 2
                inside = (x1 <= point_x < x2) and (y1 <= point_y < y2)
                if not inside or box_area >= smallest_area[row, col]:
                    continue
                smallest_area[row, col] = box_area
                objectness[row, col] = 1.0
                # зсув центра рамки ВІД центра клітинки: так голова однакова
                # для всіх клітинок і не мусить знати своїх координат
                box_target[0, row, col] = ((x1 + x2) / 2 - point_x) / (4 * CELL) + 0.5
                box_target[1, row, col] = ((y1 + y2) / 2 - point_y) / (4 * CELL) + 0.5
                box_target[2, row, col] = (x2 - x1) / SIZE
                box_target[3, row, col] = (y2 - y1) / SIZE
                class_target[row, col] = label
    return objectness, box_target, class_target


def pack(dataset):
    images = np.stack([scene[0] for scene in dataset])[:, None]
    objectness, boxes, classes = [], [], []
    for _image, box_list, label_list in dataset:
        o, b, c = encode_targets(box_list, label_list)
        objectness.append(o); boxes.append(b); classes.append(c)
    return (torch.from_numpy(images), torch.from_numpy(np.stack(objectness)),
            torch.from_numpy(np.stack(boxes)), torch.from_numpy(np.stack(classes)))


train_images, train_obj, train_box, train_cls = pack(train_set)
test_images, _, _, _ = pack(test_set)

positive_cells = int(train_obj.sum().item())
all_cells = len(train_set) * GRID * GRID
objects_in_train = sum(len(scene[1]) for scene in train_set)

print("клітинок усього        %6d" % all_cells)
print("клітинок із предметом  %6d  (%.1f %%)" % (positive_cells,
                                                 100 * positive_cells / all_cells))
print("клітинок фону          %6d  (%.1f %%)" % (all_cells - positive_cells,
                                                 100 * (1 - positive_cells / all_cells)))
print()
print("предметів на сцену     %.2f" % (objects_in_train / len(train_set)))
print("клітинок на предмет    %.2f" % (positive_cells / objects_in_train))
print()
print("якби на предмет припадала рівно одна клітинка, фону було б %.1f %%"
      % (100 * 61 / 64))
print("Модель, яка мовчки каже «предмета немає», має саме цю «точність» — і нуль користі.")

## 7 · Детектор

Маленька згорткова мережа: три блоки «згортка + батчнорм + ReLU + пулінг»
зменшують 64×64 до 8×8, а голова 1×1 віддає вісім чисел на клітинку —
обʼєктність, чотири числа рамки й три класи.

In [ ]:
class Detector(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.body = nn.Sequential(
            block(1, 16), block(16, 32), block(32, 64),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.head = nn.Conv2d(64, 8, 1)

    def forward(self, x):
        y = self.head(self.body(x))
        return y[:, 0], y[:, 1:5], y[:, 5:8]     # обʼєктність, рамка, класи


torch.manual_seed(0)
model = Detector()
print("параметрів:", sum(p.numel() for p in model.parameters()))

Навчання: 25 епох, AdamW зі швидкістю навчання 3·10⁻³, партії по 32 сцени. Втрата
складається з трьох доданків — обʼєктність по всіх клітинках, рамка й клас **лише
по позитивних**.

⏱ Ця клітинка йде 35-40 секунд.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
objectness_loss = nn.BCEWithLogitsLoss()

started = time.time()
model.train()
for epoch in range(25):
    order = torch.randperm(len(train_images))
    for start in range(0, len(train_images), 32):
        batch = order[start:start + 32]
        images = train_images[batch]
        target_obj = train_obj[batch]
        target_box = train_box[batch]
        target_cls = train_cls[batch]

        predicted_obj, predicted_box, predicted_cls = model(images)
        loss = objectness_loss(predicted_obj, target_obj)

        positive = target_obj > 0.5
        # рамку й клас вимагаємо тільки там, де предмет справді є
        box_here = torch.sigmoid(predicted_box).permute(0, 2, 3, 1)[positive]
        loss = loss + 5.0 * ((box_here - target_box.permute(0, 2, 3, 1)[positive]) ** 2).mean()
        loss = loss + nn.functional.cross_entropy(
            predicted_cls.permute(0, 2, 3, 1)[positive], target_cls[positive])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

training_seconds = time.time() - started
print("навчання зайняло %.1f с" % training_seconds)
print("остання втрата   %.4f" % loss.item())

## 8 · Що видає детектор

Голова рахує рамку **з кожної клітинки**, тобто 64 рамки на сцену незалежно від
того, скільки там предметів. Оцінкою рамки беремо добуток обʼєктності на ймовірність
обраного класу — так роблять і справжні детектори.

In [ ]:
@torch.no_grad()
def predict(images):
    """Розкодовує вихід мережі в списки рамок, оцінок і класів — по сцені на список."""
    model.eval()
    raw_obj, raw_box, raw_cls = model(images)
    confidence = torch.sigmoid(raw_obj)
    box_values = torch.sigmoid(raw_box)
    class_probability = torch.softmax(raw_cls, 1)
    class_choice = raw_cls.argmax(1)

    results = []
    for n in range(len(images)):
        boxes, scores, labels = [], [], []
        for row in range(GRID):
            for col in range(GRID):
                point_x = col * CELL + CELL / 2
                point_y = row * CELL + CELL / 2
                center_x = point_x + (box_values[n, 0, row, col].item() - 0.5) * 4 * CELL
                center_y = point_y + (box_values[n, 1, row, col].item() - 0.5) * 4 * CELL
                width = box_values[n, 2, row, col].item() * SIZE
                height = box_values[n, 3, row, col].item() * SIZE
                label = int(class_choice[n, row, col].item())

                boxes.append([center_x - width / 2, center_y - height / 2,
                              center_x + width / 2, center_y + height / 2])
                scores.append(confidence[n, row, col].item()
                              * class_probability[n, label, row, col].item())
                labels.append(label)
        results.append((torch.tensor(boxes), torch.tensor(scores), torch.tensor(labels)))
    return results


raw_predictions = predict(test_images)
ground_truth = [(torch.from_numpy(scene[1]), torch.from_numpy(scene[2]))
                for scene in test_set]

CONFIDENCE = 0.05
above_threshold = sum(int((scores >= CONFIDENCE).sum())
                      for _boxes, scores, _labels in raw_predictions)

print("рамок узагалі         %5d  (120 сцен × 64 клітинки)" % (120 * 64))
print("рамок з оцінкою ≥ %.2f %5d" % (CONFIDENCE, above_threshold))
print("справжніх предметів   %5d" % true_box_count)
print()
print("Тобто на кожен предмет припадає близько %.1f рамки. Це не помилка навчання —"
      % (above_threshold / true_box_count))
print("це кілька клітинок, які бачать той самий предмет. Далі їх прибере NMS.")

## 9 · Свій NMS проти бібліотечного

Алгоритм у чотири кроки: відсортувати за спаданням оцінки, узяти найвпевненішу,
викинути все, що перекривається з нею сильніше за поріг, повторити.

Тонкість, у якій легко помилитись: викидаємо, коли IoU **строго більший** за поріг.
Рамка з IoU рівно 0.5 при порозі 0.5 виживає — саме так поводиться `ops.nms`.

In [ ]:
def nms_by_hand(boxes, scores, threshold):
    """Повертає індекси рамок, які треба лишити."""
    order = torch.argsort(scores, descending=True).tolist()
    keep = []
    while order:
        best = order.pop(0)
        keep.append(best)
        survivors = []
        for candidate in order:
            overlap, _, _ = iou_by_hand(boxes[best].tolist(), boxes[candidate].tolist())
            if overlap <= threshold:          # строго більший — викидаємо
                survivors.append(candidate)
        order = survivors
    return keep


# звіряємо на справжніх передбаченнях: беремо кілька сцен і кілька порогів
mismatch = 0
for scene_index in range(12):
    boxes, scores, _labels = raw_predictions[scene_index]
    chosen = scores >= 0.02
    boxes, scores = boxes[chosen], scores[chosen]
    for threshold in (0.1, 0.3, 0.5, 0.7, 0.9):
        ours = nms_by_hand(boxes, scores, threshold)
        theirs = nms(boxes, scores, threshold).tolist()
        if ours != theirs:
            mismatch += 1
            print("розбіжність: сцена", scene_index, "поріг", threshold)

print("перевірено 12 сцен × 5 порогів, розбіжностей:", mismatch)
assert mismatch == 0, "наш NMS поводиться не так, як бібліотечний!"
print("✅ наш NMS дає ті самі індекси, що ops.nms")

## 10 · Що NMS робить із реальними передбаченнями

Прогонимо весь перевірний набір при різних порогах NMS і подивимось, скільки рамок
лишається. Заразом порахуємо точність і повноту — для цього потрібне зіставлення.

In [ ]:
def greedy_match(boxes, scores, truth_boxes, iou_threshold):
    """Жадібне зіставлення за спаданням упевненості.

    Повертає впорядковані оцінки й позначки «влучна / хибна» тієї самої довжини.
    Одну істинну рамку двічі не зараховуємо: інакше сто копій правильної рамки
    дали б сто влучань.
    """
    order = torch.argsort(scores, descending=True)
    taken = [False] * len(truth_boxes)
    hits = torch.zeros(len(order))

    if len(truth_boxes) and len(order):
        overlaps = box_iou(boxes[order], truth_boxes)
        for position in range(len(order)):
            best_value, best_index = -1.0, -1
            for truth_index in range(len(truth_boxes)):
                if taken[truth_index]:
                    continue
                if overlaps[position, truth_index].item() > best_value:
                    best_value = overlaps[position, truth_index].item()
                    best_index = truth_index
            if best_index >= 0 and best_value >= iou_threshold:
                taken[best_index] = True
                hits[position] = 1.0
    return scores[order], hits


def evaluate(confidence_threshold, nms_threshold, iou_threshold=0.5):
    """Точність і повнота всього набору при заданих трьох порогах."""
    all_scores, all_hits, box_count = [], [], 0
    for (boxes, scores, _labels), (truth_boxes, _truth_labels) in zip(raw_predictions,
                                                                     ground_truth):
        chosen = scores >= confidence_threshold
        boxes, scores = boxes[chosen], scores[chosen]
        if len(boxes):
            keep = nms(boxes, scores, nms_threshold)
            boxes, scores = boxes[keep], scores[keep]
        box_count += len(boxes)
        ordered_scores, hits = greedy_match(boxes, scores, truth_boxes, iou_threshold)
        all_scores.append(ordered_scores)
        all_hits.append(hits)

    scores = torch.cat(all_scores)
    hits = torch.cat(all_hits)
    precision = hits.sum().item() / max(1, len(hits))
    recall = hits.sum().item() / true_box_count
    return box_count, precision, recall, scores, hits


print("поріг NMS   рамок   точність   повнота")
for threshold in (0.1, 0.2, 0.3, 0.5, 0.7, 0.9):
    count, precision, recall, _, _ = evaluate(CONFIDENCE, threshold)
    print("   %.2f      %5d     %.3f      %.3f" % (threshold, count, precision, recall))

count, precision, recall, _, _ = evaluate(CONFIDENCE, 1.01)
print("  без NMS    %5d     %.3f      %.3f" % (count, precision, recall))

Головне тут — не окремі числа, а їхній обмін. Жадібний поріг 0.1 піднімає точність
майже до одиниці, але викидає разом із дублікатами й справжні предмети: повнота
падає. Відсутність NMS зберігає повноту, але точність стає непристойною.

## 11 · Робоча точка й ручка порога

Тепер зафіксуємо NMS на 0.5 і покрутимо **поріг упевненості**. Ваги детектора при
цьому не міняються — це буквально одна модель.

In [ ]:
NMS_THRESHOLD = 0.5
sweep = []
print("поріг      рамок   точність   повнота")
for threshold in (1e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.05, 0.1, 0.3, 0.5, 0.8, 0.95, 0.99, 0.999):
    count, precision, recall, _, _ = evaluate(threshold, NMS_THRESHOLD)
    sweep.append((threshold, count, precision, recall))
    print("%8.4f   %5d     %.3f      %.3f" % (threshold, count, precision, recall))

count, precision, recall, _, _ = evaluate(CONFIDENCE, NMS_THRESHOLD)
hits_count = round(precision * count)
print()
print("Робоча точка (поріг %.2f, NMS %.2f):" % (CONFIDENCE, NMS_THRESHOLD))
print("  %d рамок на %d істинних" % (count, true_box_count))
print("  влучних %d, хибних %d, пропущено %d"
      % (hits_count, count - hits_count, true_box_count - hits_count))
print("  точність %.3f, повнота %.3f" % (precision, recall))

## 12 · AP: площа під кривою точність-повнота

Спершу перевіримо власну реалізацію на прикладі, який можна порахувати **на
пальцях**. Три істинні предмети, пʼять передбачень; перше влучне, друге хибне,
третє влучне, четверте хибне, пʼяте влучне.

Крива йде так:

| крок | влучних | усього | точність | повнота |
|---|---|---|---|---|
| 1 | 1 | 1 | 1.000 | 0.333 |
| 2 | 1 | 2 | 0.500 | 0.333 |
| 3 | 2 | 3 | 0.667 | 0.667 |
| 4 | 2 | 4 | 0.500 | 0.667 |
| 5 | 3 | 5 | 0.600 | 1.000 |

Огинальна (найбільша точність праворуч) у трьох точках, де повнота росте, дорівнює
1.000, 0.667 і 0.600. Площа береться прирістом повноти на цю точність:

`AP = 1/3 · 1.000 + 1/3 · 0.667 + 1/3 · 0.600 = 0.7556`

In [ ]:
def average_precision(scores, hits, truth_count, method="all"):
    """AP за оцінками й позначками влучності. Два способи взяти площу."""
    order = torch.argsort(scores, descending=True)
    ordered_hits = hits[order]

    running_hits = torch.cumsum(ordered_hits, 0)
    running_misses = torch.cumsum(1 - ordered_hits, 0)
    precision = running_hits / (running_hits + running_misses)
    recall = running_hits / truth_count

    if method == "eleven":
        total = 0.0
        for level in np.arange(0, 1.001, 0.1):
            reachable = recall >= level - 1e-9
            total += precision[reachable].max().item() if reachable.any() else 0.0
        return total / 11.0

    # огинальна: у кожній точці беремо найбільшу точність, досяжну праворуч
    envelope = precision.clone()
    for i in range(len(envelope) - 2, -1, -1):
        envelope[i] = max(envelope[i].item(), envelope[i + 1].item())

    area, previous_recall = 0.0, 0.0
    for i in range(len(envelope)):
        area += (recall[i].item() - previous_recall) * envelope[i].item()
        previous_recall = recall[i].item()
    return area


tiny_scores = torch.tensor([0.9, 0.8, 0.7, 0.6, 0.5])
tiny_hits = torch.tensor([1.0, 0.0, 1.0, 0.0, 1.0])
our_value = average_precision(tiny_scores, tiny_hits, 3, "all")
by_hand = (1 / 3) * 1.0 + (1 / 3) * (2 / 3) + (1 / 3) * 0.6

print("наша функція  %.6f" % our_value)
print("руками        %.6f" % by_hand)
# порівнюємо з допуском: тензори float32 не дають точних третин
assert abs(our_value - by_hand) < 1e-6, "AP розійшовся з ручним підрахунком!"
print("✅ збігається")

Тепер те саме на справжньому детекторі. Поріг упевненості беремо якнайнижчий —
крива має пройти всі робочі точки, а не лише ту, яку ми обрали.

In [ ]:
_count, _precision, _recall, all_scores, all_hits = evaluate(1e-4, NMS_THRESHOLD, 0.5)

ap_all_points = average_precision(all_scores, all_hits, true_box_count, "all")
ap_eleven = average_precision(all_scores, all_hits, true_box_count, "eleven")

print("AP@0.5 методом усіх точок   %.4f" % ap_all_points)
print("AP@0.5 одинадцятиточковий   %.4f" % ap_eleven)
print("різниця                     %.4f" % (ap_all_points - ap_eleven))
print()
order = torch.argsort(all_scores, descending=True)
best_recall = (torch.cumsum(all_hits[order], 0) / true_box_count).max().item()
print("максимальна досяжна повнота %.4f" % best_recall)
print("Повноти 1.0 детектор не досягає ніколи, а одинадцятиточковий метод усе одно")
print("питає про неї й зараховує нуль. Один цей нуль важить 1/11 = %.4f." % (1 / 11))

Намалюємо криву разом з її огинальною — саме під огинальною береться площа.

In [ ]:
order = torch.argsort(all_scores, descending=True)
ordered_hits = all_hits[order]
running_hits = torch.cumsum(ordered_hits, 0)
running_misses = torch.cumsum(1 - ordered_hits, 0)
precision_curve = (running_hits / (running_hits + running_misses)).numpy()
recall_curve = (running_hits / true_box_count).numpy()

envelope = precision_curve.copy()
for i in range(len(envelope) - 2, -1, -1):
    envelope[i] = max(envelope[i], envelope[i + 1])

plt.figure(figsize=(6, 4))
plt.plot(recall_curve, precision_curve, linewidth=0.8, color="#93a3b4", label="сира точність")
plt.plot(recall_curve, envelope, linewidth=2, color="#c2185b", label="огинальна")
plt.fill_between(recall_curve, envelope, alpha=0.15, color="#c2185b")
plt.xlabel("повнота"); plt.ylabel("точність")
plt.title("AP@0.5 = %.4f" % ap_all_points)
plt.xlim(0, 1); plt.ylim(0, 1.02); plt.legend(loc="lower left"); plt.grid(alpha=0.25)
plt.tight_layout(); plt.show()
print("Площа під товстою лінією і є AP.")

## 13 · mAP: усереднення по класах і по порогах IoU

`mAP@0.5` — середнє AP по класах при одному порозі. `mAP@0.5:0.95` — середнє по
десяти порогах від 0.50 до 0.95 із кроком 0.05. Це один і той самий детектор, і
різниця між двома числами показує, наскільки точно він ставить рамку.

In [ ]:
def average_precision_for_class(class_index, iou_threshold):
    """AP одного класу: його рамки проти його ж істинних рамок."""
    scores_parts, hits_parts, truth_count = [], [], 0
    for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(raw_predictions,
                                                                    ground_truth):
        chosen = (scores >= 1e-4) & (labels == class_index)
        picked_boxes, picked_scores = boxes[chosen], scores[chosen]
        if len(picked_boxes):
            keep = nms(picked_boxes, picked_scores, NMS_THRESHOLD)
            picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]

        this_class_truth = truth_boxes[truth_labels == class_index]
        truth_count += len(this_class_truth)
        ordered_scores, hits = greedy_match(picked_boxes, picked_scores,
                                            this_class_truth, iou_threshold)
        scores_parts.append(ordered_scores)
        hits_parts.append(hits)

    return average_precision(torch.cat(scores_parts), torch.cat(hits_parts),
                             max(1, truth_count), "all"), truth_count


thresholds = [round(float(t), 2) for t in np.arange(0.5, 0.951, 0.05)]
grid = []
print("клас        рамок   AP@0.5   AP@0.5:0.95")
for class_index in range(3):
    row = [average_precision_for_class(class_index, t)[0] for t in thresholds]
    grid.append(row)
    _value, count = average_precision_for_class(class_index, 0.5)
    print("%-11s %5d   %.4f   %.4f" % (CLASS_NAMES[class_index], count,
                                        row[0], float(np.mean(row))))

map_50 = float(np.mean([row[0] for row in grid]))
map_50_95 = float(np.mean([np.mean(row) for row in grid]))
print()
print("mAP@0.5      = %.4f" % map_50)
print("mAP@0.5:0.95 = %.4f" % map_50_95)
print()
print("Це одна модель на тих самих 120 сценах. Від першого числа до другого")
print("зникло %.1f %% — саме стільки коштує вимога ставити рамку точно."
      % (100 * (map_50 - map_50_95) / map_50))

Подивимось, як AP спадає з ростом порога IoU. Наш детектор передбачає рамку з
клітинки 8×8 пікселів, тому 2-3 пікселі похибки для нього норма — і при порозі 0.9
від AP не лишається майже нічого.

In [ ]:
print("поріг IoU   коло    квадрат  трикутник  середнє")
for column, threshold in enumerate(thresholds):
    values = [grid[c][column] for c in range(3)]
    print("   %.2f     %.4f  %.4f   %.4f     %.4f"
          % (threshold, values[0], values[1], values[2], float(np.mean(values))))

plt.figure(figsize=(6, 3.6))
for class_index in range(3):
    plt.plot(thresholds, grid[class_index], marker="o", label=CLASS_NAMES[class_index])
plt.plot(thresholds, [float(np.mean([grid[c][i] for c in range(3)]))
                      for i in range(len(thresholds))],
         color="black", linewidth=2, label="mAP")
plt.xlabel("поріг IoU"); plt.ylabel("AP"); plt.grid(alpha=0.25); plt.legend()
plt.tight_layout(); plt.show()
print("Чим суворіший поріг, тим менше лишається від тієї самої моделі.")

## 14 · Де NMS ламається

Досі NMS виглядав корисним. Ось випадок, у якому він робить непоправну шкоду: два
**різні справжні** предмети, що перекриваються, — машина за машиною, людина за
людиною. Рамки 60 на 120 пікселів, задня зсунута вбік на кілька пікселів.

In [ ]:
print("зсув     IoU     поріг 0.3    поріг 0.5    поріг 0.7")
for shift in (10, 15, 20, 25):
    pair = torch.tensor([[0.0, 0.0, 60.0, 120.0],
                         [float(shift), 0.0, 60.0 + shift, 120.0]])
    confidences = torch.tensor([0.90, 0.85])
    overlap = box_iou(pair[:1], pair[1:])[0, 0].item()

    verdicts = []
    for threshold in (0.3, 0.5, 0.7):
        survivors = len(nms(pair, confidences, threshold))
        verdicts.append("обидві   " if survivors == 2 else "ЗАГУБИЛИ ")
    print("%2d px   %.3f    %s    %s    %s" % (shift, overlap, *verdicts))

print()
print("При загальноприйнятому порозі 0.5 два предмети, що стоять на відстані")
print("10 або 15 пікселів, перетворюються на один: NMS викидає задній назавжди.")
print("Жоден наступний етап його не поверне — у метриці це буде звичайний пропуск.")

Підняти поріг теж не вихід: ми вже бачили в розділі 10, що при NMS 0.9 точність
падає нижче за 0.25. Низький поріг губить справжні предмети, високий лишає
дублікати, і геометрія не дає способу відрізнити одне від одного. Часткове
лікування — **Soft-NMS**: перекритій рамці не викидають, а знижують упевненість
тим сильніше, чим більший IoU. Радикальне — прибрати NMS зовсім, як це робить
DETR, про який ітиметься в темі 27.

## 15 · Підсумок зошита

In [ ]:
print("Що поміряно:")
print()
print("  IoU руками                     %.6f  (box_iou: те саме)" % value)
print("  клітинок фону                  %.1f %%" % (100 * (1 - positive_cells / all_cells)))
print("  навчання детектора             %.1f с" % training_seconds)
print("  рамок до NMS                   %d на %d предметів" % (above_threshold,
                                                               true_box_count))
count, precision, recall, _, _ = evaluate(CONFIDENCE, NMS_THRESHOLD)
print("  рамок після NMS                %d, точність %.3f, повнота %.3f"
      % (count, precision, recall))
print("  AP@0.5 усі точки               %.4f" % ap_all_points)
print("  AP@0.5 одинадцять точок        %.4f" % ap_eleven)
print("  mAP@0.5                        %.4f" % map_50)
print("  mAP@0.5:0.95                   %.4f" % map_50_95)
print()
print("Головна пара чисел теми — два останні рядки. Це одна модель.")

## Завдання

### 🟢 Рівень 1 — База

Зміни `CONFIDENCE` на 0.3 і 0.001 і перерахуй робочу точку. Побудуй графік
«точність і повнота проти порога» за списком `sweep`.

**Зроблено, якщо:** графік показує, що зі зниженням порога повнота росте, а
точність падає, і ти можеш назвати поріг, при якому точність уперше опускається
нижче за 0.5.

### 🟡 Рівень 2 — Плюс

Порахуй AP окремо для **великих** і **малих** предметів: розділи істинні рамки за
площею на дві половини й пропусти кожну групу через `average_precision`.

**Зроблено, якщо:** ти дістав два числа й можеш пояснити, чому вони різні —
скориставшись таблицею допустимого зсуву з розділу 3.

### 🔴 Рівень 3 — Виклик

Реалізуй **Soft-NMS**: замість викидати перекриту рамку, помнож її оцінку на
`exp(−IoU² / σ)` з `σ = 0.5`, а потім відкинь усе, що впало нижче за 0.05.
Порівняй mAP@0.5 звичайного NMS і Soft-NMS на нашому перевірному наборі.

**Зроблено, якщо:** обидва числа надруковані, і ти можеш сказати, за рахунок чого
змінилась різниця — точності чи повноти.